# 🚀 Модуль 6. Урок 1. Архитектура BI и подключение источников данных в Looker Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_06_bi_looker_studio/lesson_01_looker_studio_setup_and_data_sources.ipynb)

## 📌 Бизнес-контекст и методология
Дашборд — это инструмент принятия управленческих решений.
В этом уроке мы разберем:
1. Архитектуру подключения BigQuery / PostgreSQL / Google Sheets.
2. Подготовку плоских денормализованных витрин (Data Marts).
3. Принцип 5 секунд (5-second rule) для верхнеуровневых Executive KPI.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd

df_saas = pd.read_csv(saas_csv)

total_mrr = df_saas['mrr'].sum()
active_clients = len(df_saas[df_saas['is_churned'] == 0])
churned_clients = len(df_saas[df_saas['is_churned'] == 1])
churn_rate = churned_clients / len(df_saas)
arpu = total_mrr / active_clients if active_clients > 0 else 0

print("=" * 45)
print("📊 EXECUTIVE SUMMARY (Верхний ярус дашборда):")
print(f"💰 Total MRR:          ${total_mrr:,.2f}")
print(f"👥 Active Clients:     {active_clients}")
print(f"🚪 Churn Rate:         {churn_rate:.2%}")
print(f"📈 ARPU (Active):      ${arpu:.2f}")
print("=" * 45)


📊 EXECUTIVE SUMMARY (Верхний ярус дашборда):
💰 Total MRR:          $158,010.00
👥 Active Clients:     1664
🚪 Churn Rate:         16.80%
📈 ARPU (Active):      $94.96


### 🎯 Практическое задание 1: Расчет дельты к целевому показателю (Target Delta)
Целевой Churn Rate в компании установлен на уровне 4.0%.
Рассчитайте абсолютную разницу между фактическим и целевым оттоком в процентных пунктах (p.p.).

In [3]:
### ВАШ КОД ЗДЕСЬ
target_churn = 0.04
delta_pp = (churn_rate - target_churn) * 100

print(f"Отклонение от цели: {delta_pp:+.2f} п.п.")
assert isinstance(delta_pp, float), "Дельта должна быть вещественным числом"
print("✅ Тест успешно пройден: метрика отклонения готова для KPI-карточки!")


Отклонение от цели: +12.80 п.п.
✅ Тест успешно пройден: метрика отклонения готова для KPI-карточки!


### 💡 Архитектурный фреймворк дашборда
* **Tier 1 (Верх)**: 3-5 ключевых KPI с дельтами к прошлому периоду и таргету.
* **Tier 2 (Середина)**: Тренды во времени (динамика недели/месяцы).
* **Tier 3 (Низ)**: Детализирующая таблица с возможностью фильтрации.